# 04 — Klasifikimi dhe regresioni logjistik

        **Pyetja udhëheqëse:** Si dallojmë sinjalin nga sfondi dhe si zgjedhim pragun e vendimit?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të interpretohet regresioni logjistik probabilistikisht.
- Të implementohet sigmoid + cross-entropy.
- Të përdoren confusion matrix, ROC, AUC dhe precision-recall.
- Të kuptohet roli i pragut.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("04 — Klasifikimi dhe regresioni logjistik")
print("Seed:", RANDOM_STATE)

# 1. Sinjal kundrejt sfondit në një detektor — CORE

Regresioni logjistik modelon
$$
P(y=1|\mathbf x)=\sigma(\mathbf w^T\mathbf x+b),\qquad
\sigma(z)=\frac{1}{1+e^{-z}}.
$$
Klasifikimi final kërkon një prag $t$: deklarojmë sinjal nëse $P(y=1|x)\ge t$.

In [ ]:
n0, n1 = 500, 260
X0 = rng.normal([0.0, 0.0], [1.0, 1.1], size=(n0,2))
X1 = rng.normal([1.8, 1.5], [0.9, 0.8], size=(n1,2))
X = np.vstack([X0, X1]); y = np.r_[np.zeros(n0), np.ones(n1)]

fig, ax = plt.subplots()
ax.scatter(X0[:,0], X0[:,1], s=10, label="sfond")
ax.scatter(X1[:,0], X1[:,1], s=10, label="sinjal")
ax.set_xlabel("Veçoria 1"); ax.set_ylabel("Veçoria 2"); ax.legend()
ax.set_title("Ngjarje detektori")
plt.show()

# 2. Sigmoid dhe funksioni i humbjes — CORE

In [ ]:
z = np.linspace(-8, 8, 400)
sigmoid = lambda z: 1/(1+np.exp(-z))
fig, ax = plt.subplots()
ax.plot(z, sigmoid(z))
ax.axhline(0.5, linestyle="--"); ax.axvline(0, linestyle="--")
ax.set_xlabel("z = w^T x + b"); ax.set_ylabel("σ(z)")
ax.set_title("Funksioni sigmoid")
plt.show()

Për një mostër binare, entropia e kryqëzuar (cross-entropy) është
$$
\ell=-\left[y\log p+(1-y)\log(1-p)\right].
$$
Ajo penalizon fort parashikimet shumë të sigurta por të gabuara.

# 3. Zbatim manual me zbritje gradienti — CORE

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)
sc = StandardScaler().fit(Xtr)
Xs, Xts = sc.transform(Xtr), sc.transform(Xte)

A = np.c_[np.ones(len(Xs)), Xs]
w = np.zeros(A.shape[1])
loss_hist = []
lr = 0.15
for _ in range(1000):
    p = sigmoid(A@w)
    grad = A.T@(p-ytr)/len(ytr)
    w -= lr*grad
    eps=1e-12
    loss_hist.append(-(ytr*np.log(p+eps)+(1-ytr)*np.log(1-p+eps)).mean())

fig, ax = plt.subplots()
ax.plot(loss_hist)
ax.set_xlabel("Iteracioni"); ax.set_ylabel("Cross-entropy")
ax.set_title("Konvergjenca e trajnimit")
plt.show()

### Krahasim me implementimin kanonik në `src/`

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))
from src.classification.logistic import LogisticRegressionGD
log_src = LogisticRegressionGD(lr=0.15, n_iter=1000).fit(Xs, ytr)
print("Peshat nga src:", np.round(log_src.weights_, 4))

# 4. Krahasim me scikit-learn dhe kufiri i vendimit — CORE

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression().fit(Xs, ytr)
proba = clf.predict_proba(Xts)[:,1]

xx, yy = np.meshgrid(np.linspace(Xs[:,0].min()-1, Xs[:,0].max()+1, 250),
                     np.linspace(Xs[:,1].min()-1, Xs[:,1].max()+1, 250))
P = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)

fig, ax = plt.subplots()
ax.contour(xx, yy, P, levels=[0.5])
ax.scatter(Xs[ytr==0,0], Xs[ytr==0,1], s=8)
ax.scatter(Xs[ytr==1,0], Xs[ytr==1,1], s=8)
ax.set_title("Kufiri i vendimit për t=0.5")
ax.set_xlabel("x1 i standardizuar"); ax.set_ylabel("x2 i standardizuar")
plt.show()

# 5. Confusion matrix dhe pragu — CORE

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
for t in [0.2, 0.5, 0.8]:
    pred=(proba>=t).astype(int)
    print(t, "precision=", round(precision_score(yte,pred),3),
          "recall=", round(recall_score(yte,pred),3),
          "F1=", round(f1_score(yte,pred),3))
ConfusionMatrixDisplay.from_predictions(yte, (proba>=0.5).astype(int))
plt.title("Matrica e konfuzionit, t=0.5")
plt.show()

# 6. ROC, AUC dhe precision-recall — CORE

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
fpr,tpr,_ = roc_curve(yte,proba)
auc=roc_auc_score(yte,proba)
prec,rec,_ = precision_recall_curve(yte,proba)

fig, ax = plt.subplots()
ax.plot(fpr,tpr,label=f"AUC={auc:.3f}")
ax.plot([0,1],[0,1],linestyle="--")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.legend(); ax.set_title("Kurba ROC")
plt.show()

fig, ax = plt.subplots()
ax.plot(rec,prec)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Kurba precision–recall")
plt.show()

**Interpretim.** ROC përmbledh kompromisin TPR–FPR në të gjitha pragjet. Precision–recall bëhet veçanërisht i vlefshëm kur klasa pozitive është e rrallë.

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 7. Si ndryshon pragu optimal? — EXTENDED

In [ ]:
thresholds=np.linspace(0.05,0.95,50)
Ps,Rs,F1s=[],[],[]
for t in thresholds:
    pred=(proba>=t).astype(int)
    Ps.append(precision_score(yte,pred,zero_division=0))
    Rs.append(recall_score(yte,pred,zero_division=0))
    F1s.append(f1_score(yte,pred,zero_division=0))
fig, ax = plt.subplots()
ax.plot(thresholds,Ps,label="precision")
ax.plot(thresholds,Rs,label="recall")
ax.plot(thresholds,F1s,label="F1")
ax.set_xlabel("Pragu"); ax.legend(); ax.set_title("Pragu ndryshon kompromisin")
plt.show()

# 8. Aplikim jo-fizik: diagnozë mjekësore — EXTENDED

Në screening mjekësor kostoja e një false negative mund të jetë shumë më e lartë se ajo e një false positive; prandaj pragu nuk duhet zgjedhur vetëm për të maksimizuar accuracy.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Llogaritni specificity për pragjet 0.2, 0.5 dhe 0.8.
2. Gjeni pragun që jep recall ≥ 0.95 me precision maksimal.
3. Shpjegoni interpretimin probabilistik të AUC.
4. Krahasoni ROC me precision-recall për një klasë pozitive 5%.

        ## B. Ushtrim i plotë
        Krijoni një problem të pabalancuar 95:5, trajnojeni regresionin logjistik dhe diskutoni pse accuracy mund të jetë mashtruese.

        ## C. Playground
        Ndryshoni distancën ndërmjet klasave, pabalancimin dhe pragun.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.